# Example 1.1 / 2: Set Expressions

Companion notebook for **1.1 Sets** in the AMPL lab Documentation, section "Set Expressions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Set Expressions

Wherever AMPL expects a set, any **set expression** may be written, not only a set's name:

| Expression | Meaning |
|---|---|
| `{"bands", "coils", "plate"}` | a set written out member by member |
| `1..12`, `1..12 by 3` | the numbers 1 to 12; every third one |
| `A union B`, `A inter B`, `A diff B`, `A symdiff B` | either set; both; in `A` but not `B`; in exactly one |
| `ORIG cross DEST` | all pairs `(i,j)` with `i` in `ORIG` and `j` in `DEST` |
| `{p in PROD: cost[p] < 10}` | the members that satisfy a condition |
| `{}` | the empty set |

Here `PROD_A` and `CHEAP` are defined in the model by set expressions. From Python, a defined set
such as `CHEAP` is read like any other set, and `ampl.get_data("...")` evaluates any set
expression, returning an amplpy `DataFrame` (`to_list()` turns it into a list). Call it once per
expression: `get_data` fails when its results are indexed over different sets.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param cost {PROD} >= 0;
set PROD_A = {"bands", "coils"};
set CHEAP = {p in PROD: cost[p] < 10};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate", "rods"]
ampl.param["cost"] = {"bands": 8, "coils": 12, "plate": 9, "rods": 25}

In [5]:
# run.py
print(ampl.set["CHEAP"].to_list())
print(ampl.get_data("PROD diff PROD_A").to_list())
print(ampl.get_data("PROD_A inter CHEAP").to_list())
print(ampl.get_data("1..12 by 3").to_list())

['bands', 'plate']
['plate', 'rods']
['bands']
[1, 4, 7, 10]


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. One `display` command can show several set
expressions:

In [6]:
%%writefile data.dat
set PROD := bands coils plate rods;
param cost := bands 8  coils 12  plate 9  rods 25;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display CHEAP, PROD diff PROD_A, PROD_A inter CHEAP, 1..12 by 3;

set CHEAP := bands plate;

set PROD diff PROD_A := plate rods;

set PROD_A inter CHEAP := bands;

set 1 .. 12 by 3 := 1 4 7 10;

